# Lab 30 — MCP + A2A composition

> ⏱ 130-160 min · 🟡 Intermediate

The closer for Path 04. Build the canonical composition pattern end-to-end: an MCP server with two tools, an A2A worker whose `execute()` opens an `MCPClient` inside the request lifetime, and an orchestrator that uses the SDK's `A2ACardResolver` + `ClientFactory` to discover and call the worker. Then layer push notifications with atomic registration and a webhook receiver.

**Prerequisites**: [Lab 29 (production depth)](../29-a2a-endpoint-production-depth/), [Lab 25 (MCP server)](../25-mcp-server-from-scratch/), [Lab 26 (MCP client)](../26-mcp-client-from-scratch/), [Module 7](../../concepts/tools/mcp-a2a-composition.md).

**Strategy**: three subprocesses (MCP server on :9991, A2A worker on :9990, webhook receiver on :9989). The notebook is the orchestrator — discovers the worker, dispatches tasks, registers push notifications, inspects webhook callbacks. The composition is asymmetric on purpose: orchestrator knows only A2A; worker knows both A2A and MCP.

## Step 0 — Environment setup

Three SDK dependencies plus the standard HTTP plumbing. Same Python environment as Lab 29 plus `fastmcp>=3.3`.

In [ ]:
import sys
import subprocess
import time
import json
from pathlib import Path

LAB_DIR = Path.cwd()
print(f"Lab directory: {LAB_DIR}")

import_failures = []
for mod_name, install_hint in [
    ("a2a", "pip install 'a2a-sdk>=1.0,<2.0'"),
    ("fastmcp", "pip install 'fastmcp>=3.3'"),
    ("httpx", "pip install httpx"),
    ("uvicorn", "pip install uvicorn"),
    ("starlette", "pip install starlette"),
]:
    try:
        __import__(mod_name)
        print(f"  ✓ {mod_name}")
    except ImportError:
        import_failures.append((mod_name, install_hint))
        print(f"  ✗ {mod_name} — {install_hint}")

if import_failures:
    print("\nFix the failures above and re-run this cell.")
    sys.exit(1)

# Versions, for the record
import a2a as _a2a
import fastmcp as _fm
print(f"\n  a2a-sdk: {getattr(_a2a, '__version__', '?')}")
print(f"  fastmcp: {_fm.__version__}")
print("\n✓ Environment ready. No API keys or external services required.")

## Step 1 — Author `knowledge_base_server.py` (MCP server)

A minimal FastMCP server with two tools: `lookup_customer` returns fake KB records keyed by ID; `summarize_request` returns a 1-line summary. Both are pure-Python — no external dependencies. HTTP transport on port 9991.

This is the *agent-to-tool* side of the composition. The worker (Step 2) will treat this server as one of its internal tools. The orchestrator (Steps 5-7) will never see it.

In [ ]:
KB_SERVER_PATH = LAB_DIR / "knowledge_base_server.py"

KB_SERVER_PATH.write_text(r'''"""Lab 30 knowledge-base MCP server."""
from fastmcp import FastMCP

mcp = FastMCP("knowledge-base")


@mcp.tool
def lookup_customer(customer_id: str) -> dict:
    """Look up customer info by ID. Returns name, tier, since-year."""
    db = {
        "C-100": {"name": "Acme Corp", "tier": "enterprise", "since": "2022"},
        "C-101": {"name": "Beta Inc", "tier": "starter", "since": "2024"},
        "C-102": {"name": "Gamma LLC", "tier": "pro", "since": "2023"},
    }
    return db.get(customer_id, {"error": f"Customer {customer_id} not found"})


@mcp.tool
def summarize_request(text: str) -> str:
    """Return a 1-line summary of the request text."""
    truncated = text[:80] + ("..." if len(text) > 80 else "")
    return f"Request summary: {truncated}"


if __name__ == "__main__":
    mcp.run(transport="http", host="127.0.0.1", port=9991)
''')
print(f"Wrote {KB_SERVER_PATH} ({KB_SERVER_PATH.stat().st_size} bytes)")
print("Tools: lookup_customer, summarize_request")
print("Transport: http on 127.0.0.1:9991")

## Step 2 — Author `composed_worker.py` (A2A worker with MCP inside)

The worker exposes a single A2A skill (`customer-process`) but uses two MCP tools to implement it. The `MCPClient` is opened inside `execute()` — one client per request. The A2A Agent Card describes only the *capability*, never the MCP implementation. That's the encapsulation that the composition pattern provides.

Notice three things in the executor:

1. The `MCPClient` URL is the worker's internal config (no orchestrator knows it).
2. Both MCP tool calls happen inside `async with MCPClient(...)` — one open, two calls, one close.
3. The result is a single A2A artifact combining both MCP results into a structured text response. The orchestrator gets text; it doesn't know the text came from two separate MCP calls.

In [ ]:
WORKER_PATH = LAB_DIR / "composed_worker.py"

WORKER_PATH.write_text(r'''"""Lab 30 A2A worker — uses MCP server internally for tool calls."""
import os
from a2a.types import AgentCard, AgentSkill, AgentCapabilities, AgentInterface
from a2a.server.agent_execution import AgentExecutor
from a2a.server.tasks import (
    InMemoryTaskStore, TaskUpdater,
    InMemoryPushNotificationConfigStore, BasePushNotificationSender,
)
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.routes import create_agent_card_routes, create_jsonrpc_routes
from a2a.helpers import new_text_part, new_task_from_user_message
from starlette.applications import Starlette
from fastmcp import Client as MCPClient
import httpx
import uvicorn


MCP_URL = os.environ.get("MCP_URL", "http://127.0.0.1:9991/mcp")


class ComposedAgent(AgentExecutor):
    """A2A worker; uses MCP server internally for tool calls."""

    async def execute(self, context, event_queue):
        text = ""
        if context.message and context.message.parts:
            for p in context.message.parts:
                if p.HasField("text"):
                    text += p.text

        task = new_task_from_user_message(context.message)
        await event_queue.enqueue_event(task)
        updater = TaskUpdater(event_queue, task.id, task.context_id)
        await updater.start_work()

        # The composition: open MCP client inside the A2A request lifetime
        async with MCPClient(MCP_URL) as mcp:
            summary = (await mcp.call_tool("summarize_request", {"text": text})).data

            # Toy intent routing: look for "C-NNN" customer IDs in the input
            customer_id = None
            for token in text.split():
                if token.startswith("C-") and len(token) >= 3:
                    customer_id = token.rstrip(".,!?;:")
                    break

            if customer_id:
                kb_result = (await mcp.call_tool(
                    "lookup_customer", {"customer_id": customer_id}
                )).data
                response = f"{summary}\nCustomer lookup ({customer_id}): {kb_result}"
            else:
                response = f"{summary}\nNo customer ID found in request."

        await updater.add_artifact(parts=[new_text_part(response)], name="result")
        await updater.complete()

    async def cancel(self, context, event_queue):
        raise NotImplementedError


def build_app():
    card = AgentCard(
        name="composed-worker",
        description="Processes customer requests with summarization and KB lookup",
        version="1.0.0",
        capabilities=AgentCapabilities(streaming=False, push_notifications=True),
        supported_interfaces=[AgentInterface(
            protocol_binding="JSONRPC", url="http://127.0.0.1:9990"
        )],
        default_input_modes=["text/plain"],
        default_output_modes=["text/plain"],
        skills=[AgentSkill(
            id="customer-process",
            name="Customer request processor",
            description="Summarizes requests and looks up customer info",
            tags=["customer"],
            input_modes=["text/plain"], output_modes=["text/plain"],
        )],
    )
    push_store = InMemoryPushNotificationConfigStore()
    push_sender = BasePushNotificationSender(
        httpx_client=httpx.AsyncClient(),
        config_store=push_store,
    )
    handler = DefaultRequestHandler(
        agent_executor=ComposedAgent(),
        task_store=InMemoryTaskStore(),
        agent_card=card,
        push_config_store=push_store,
        push_sender=push_sender,
    )
    return Starlette(
        routes=create_agent_card_routes(agent_card=card) +
               create_jsonrpc_routes(request_handler=handler, rpc_url="/"),
    )


if __name__ == "__main__":
    uvicorn.run(build_app(), host="127.0.0.1", port=9990, log_level="warning")
''')
print(f"Wrote {WORKER_PATH} ({WORKER_PATH.stat().st_size} bytes)")
print("Worker skill: customer-process (MCP-backed)")
print("Capabilities: push_notifications=True (for Step 7)")
print("Reads MCP from: http://127.0.0.1:9991/mcp")

## Step 3 — Author `webhook_receiver.py`

A tiny Starlette app that accepts POSTs on `/`, captures the body + headers into `notifications.jsonl`, and returns 200. The notebook reads back the file in Step 7 to verify the webhook actually fired.

The X-A2A-Notification-Token header is the HMAC shared secret per the spec. Production receivers MUST validate it before treating the body as authoritative. The lab demonstrates the header is present; building HMAC validation around it is the natural follow-on.

In [ ]:
WEBHOOK_PATH = LAB_DIR / "webhook_receiver.py"

WEBHOOK_PATH.write_text(r'''"""Lab 30 webhook receiver — captures push notifications."""
import json
import os
from starlette.applications import Starlette
from starlette.responses import JSONResponse
from starlette.routing import Route
import uvicorn

NOTIFICATIONS_PATH = os.environ.get(
    "WEBHOOK_NOTIFICATIONS_PATH", "./notifications.jsonl"
)


async def receive(request):
    body = await request.body()
    with open(NOTIFICATIONS_PATH, "a") as f:
        f.write(json.dumps({
            "headers": dict(request.headers),
            "body": body.decode(errors="replace"),
        }) + "\n")
    return JSONResponse({"ok": True})


app = Starlette(routes=[Route("/", receive, methods=["POST"])])


if __name__ == "__main__":
    uvicorn.run(app, host="127.0.0.1", port=9989, log_level="warning")
''')
print(f"Wrote {WEBHOOK_PATH} ({WEBHOOK_PATH.stat().st_size} bytes)")
print("Listens on 127.0.0.1:9989, POSTs append to notifications.jsonl")

## Step 4 — Start all three subprocesses; health-probe each

Three uvicorn processes:
- MCP server on `:9991` (fastmcp's built-in transport)
- A2A worker on `:9990` (uvicorn)
- Webhook receiver on `:9989` (uvicorn)

The health probes verify each is responding before the orchestrator-side tests start. If a port is already in use, kill stray processes and re-run.

In [ ]:
NOTIFICATIONS_PATH = LAB_DIR / "notifications.jsonl"
# Clean any leftover notification file from prior runs
if NOTIFICATIONS_PATH.exists():
    NOTIFICATIONS_PATH.unlink()

# Spawn the three subprocesses
mcp_proc = subprocess.Popen(
    [sys.executable, str(KB_SERVER_PATH)],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE,
)
worker_proc = subprocess.Popen(
    [sys.executable, str(WORKER_PATH)],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE,
)
webhook_proc = subprocess.Popen(
    [sys.executable, str(WEBHOOK_PATH)],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE,
)

# Give them time to bind
time.sleep(3.0)

print(f"  MCP server   PID {mcp_proc.pid}")
print(f"  A2A worker   PID {worker_proc.pid}")
print(f"  Webhook recv PID {webhook_proc.pid}")

In [ ]:
import httpx
import asyncio
from fastmcp import Client as MCPClient

# 4a — MCP health probe via fastmcp.Client
async def probe_mcp():
    async with MCPClient("http://127.0.0.1:9991/mcp") as c:
        tools = await c.list_tools()
        return [t.name for t in tools]

mcp_tools = await probe_mcp()
print(f"✓ MCP server up; tools: {mcp_tools}")
assert "lookup_customer" in mcp_tools
assert "summarize_request" in mcp_tools

# 4b — A2A worker health probe via Agent Card discovery
for _attempt in range(5):
    try:
        r = httpx.get("http://127.0.0.1:9990/.well-known/agent-card.json", timeout=2.0)
        if r.status_code == 200:
            break
    except (httpx.ConnectError, httpx.ReadTimeout):
        time.sleep(0.5)
else:
    raise RuntimeError(f"A2A worker not responding: {worker_proc.stderr.read().decode()[:1000]}")
worker_card = r.json()
print(f"✓ A2A worker up: {worker_card['name']}")

# 4c — Webhook receiver health probe via a direct POST
r = httpx.post("http://127.0.0.1:9989/", json={"ping": True}, timeout=2.0)
assert r.status_code == 200
print(f"✓ Webhook receiver up: status {r.status_code}")

# Confirm the ping landed in the JSONL file
assert NOTIFICATIONS_PATH.exists()
with open(NOTIFICATIONS_PATH) as f:
    lines = f.read().splitlines()
print(f"  notifications.jsonl has {len(lines)} entry (the health-probe ping)")

# Reset notifications for the real push test in Step 7
NOTIFICATIONS_PATH.unlink()

## Step 5 — Orchestrator discovery via `A2ACardResolver`

The SDK ships `A2ACardResolver` for client-side discovery. Pass an `httpx.AsyncClient` (so the connection can be pooled across multiple resolutions) and a `base_url`; call `get_agent_card()`; get back a protobuf `AgentCard`.

The point of inspecting the card here is to verify what the card *doesn't* say. The card describes the worker's `customer-process` skill — what it does, what tags apply, what input/output modes it accepts. It does NOT mention the MCP server, the KB tools, or the internal implementation. The capability is the public contract; the implementation is private.

In [ ]:
from a2a.client import A2ACardResolver

async def discover_worker():
    async with httpx.AsyncClient(timeout=10.0) as http:
        resolver = A2ACardResolver(httpx_client=http, base_url="http://127.0.0.1:9990")
        card = await resolver.get_agent_card()
        return card

card = await discover_worker()
print(f"Worker: {card.name} v{card.version}")
print(f"Description: {card.description}")
print("\nCapabilities:")
print(f"  streaming: {card.capabilities.streaming}")
print(f"  push_notifications: {card.capabilities.push_notifications}")
print("\nSkills declared:")
for skill in card.skills:
    print(f"  • {skill.id}: {skill.name}")
    print(f"    description: {skill.description}")
    print(f"    tags: {list(skill.tags)}")

# Encapsulation check
card_str = str(card)
print(f"\nDoes the card mention MCP anywhere? {'mcp' in card_str.lower() or 'MCP' in card_str}")
print(f"Does the card expose port 9991? {'9991' in card_str}")
print("  → Good encapsulation: card describes capability, not implementation")

## Step 6 — Orchestrator → composed worker via `ClientFactory`

`ClientFactory(config=ClientConfig(...))` is the SDK's high-level client builder. Pass it the resolved card; get back a `Client` instance with `send_message`, `get_task`, `subscribe`, `create_task_push_notification_config`, and the other client-side methods.

`client.send_message(SendMessageRequest(message=msg))` returns an *async iterator* of `StreamResponse` events. With `streaming=False` in the config (the default for synchronous requests), the iterator yields a single final event containing the completed task.

The result text in the artifact will combine the summarization output and the customer-lookup output — both coming from MCP tools that ran inside the worker's `execute()`. The orchestrator never knew the MCP server existed.

In [ ]:
from a2a.client import ClientFactory, ClientConfig
from a2a.types import SendMessageRequest, Message, Role
from a2a.helpers import new_text_part
from google.protobuf.json_format import MessageToDict


async def dispatch_task(card, prompt):
    async with httpx.AsyncClient(timeout=15.0) as http:
        # Build the client from the discovered card
        config = ClientConfig(httpx_client=http, streaming=False)
        factory = ClientFactory(config=config)
        client = factory.create(card)

        # Build and send the message
        msg = Message(
            message_id="orch-step6",
            role=Role.ROLE_USER,
            parts=[new_text_part(prompt)],
        )
        request = SendMessageRequest(message=msg)

        # The send_message call returns an async iterator
        async for event in client.send_message(request):
            d = MessageToDict(event)
            if "task" in d and d["task"]["status"]["state"] == "TASK_STATE_COMPLETED":
                return d["task"]
        raise RuntimeError("no completed task in response")


task = await dispatch_task(card, "Please review customer C-100 for the quarterly check-in")
print(f"Task ID: {task['id']}")
print(f"State: {task['status']['state']}")
print("\nResult artifact:")
for line in task["artifacts"][0]["parts"][0]["text"].splitlines():
    print(f"  {line}")
print("\n  → Orchestrator received an A2A artifact whose content was sourced from two MCP tool calls inside the worker.")
print("  → Orchestrator never talked to MCP directly.")

## Step 7 — Push notifications with atomic registration

The deferral from Module 6 closes here. Three coordinated pieces:

1. **`TaskPushNotificationConfig`** — names the webhook URL + a shared secret token.
2. **`SendMessageConfiguration(task_push_notification_config=..., return_immediately=True)`** — wraps the config so it's registered atomically with the message send. The `return_immediately=True` flag makes A2A actually async — `send_message` returns the SUBMITTED task without waiting for completion.
3. **The webhook receiver** — captures the resulting POSTs to `notifications.jsonl`.

The alternative pattern (calling `client.create_task_push_notification_config()` separately after sending the message) races with task completion: if the task finishes before the standalone registration call completes, no notification fires. The atomic pattern doesn't race.

In [ ]:
from a2a.types import (
    SendMessageConfiguration, TaskPushNotificationConfig,
)


async def dispatch_with_push(card, prompt, webhook_url, token):
    async with httpx.AsyncClient(timeout=15.0) as http:
        factory = ClientFactory(config=ClientConfig(httpx_client=http, streaming=False))
        client = factory.create(card)

        push_cfg = TaskPushNotificationConfig(url=webhook_url, token=token)
        config = SendMessageConfiguration(
            task_push_notification_config=push_cfg,
            return_immediately=True,
        )
        msg = Message(
            message_id="orch-step7",
            role=Role.ROLE_USER,
            parts=[new_text_part(prompt)],
        )

        async for event in client.send_message(
            SendMessageRequest(message=msg, configuration=config)
        ):
            d = MessageToDict(event)
            if "task" in d:
                return d["task"]
        raise RuntimeError("no task in response")


task = await dispatch_with_push(
    card,
    "Please review customer C-101 for the contract renewal",
    webhook_url="http://127.0.0.1:9989/",
    token="lab30-shared-secret",
)
print(f"Submitted task: {task['id']}")
print(f"State at submission: {task['status']['state']}")
print("  → Orchestrator did NOT wait for completion. Server still processing.")

In [ ]:
# Poll the notifications file until callbacks arrive

received = []
for attempt in range(20):
    await asyncio.sleep(0.5)
    if NOTIFICATIONS_PATH.exists():
        with open(NOTIFICATIONS_PATH) as f:
            received = f.read().splitlines()
        if received:
            print(f"  After {(attempt+1)*0.5:.1f}s: {len(received)} notification(s)")
            # Stop polling once we have multiple (artifact + status)
            if len(received) >= 2:
                break
else:
    print("  ⚠ No notifications received within 10s")

assert received, "Expected at least one push notification"
print(f"\n✓ Received {len(received)} push notification(s) from the worker")

In [ ]:
# Inspect what the receiver captured

print("━━━ Notification details ━━━\n")
for i, line in enumerate(received):
    notif = json.loads(line)
    headers = notif["headers"]
    body = json.loads(notif["body"])

    print(f"Notification #{i+1}")
    print(f"  Content-Type:                {headers.get('content-type')}")
    print(f"  X-A2A-Notification-Token:    {headers.get('x-a2a-notification-token')}")

    if "task" in body:
        t = body["task"]
        artifacts = t.get("artifacts", [])
        print(f"  Body kind: full task (id={t['id'][:8]}..., state={t['status']['state']})")
        if artifacts:
            print(f"  Artifact: {artifacts[0]['parts'][0]['text'][:100]}")
    elif "statusUpdate" in body:
        s = body["statusUpdate"]
        print(f"  Body kind: statusUpdate (state={s['status']['state']})")
    elif "artifactUpdate" in body:
        a = body["artifactUpdate"]
        print(f"  Body kind: artifactUpdate (artifact={a['artifact']['name']})")
    print()

# Token validation check — production receivers MUST do this
expected_token = "lab30-shared-secret"
received_tokens = {json.loads(n)["headers"].get("x-a2a-notification-token") for n in received}
print(f"  Token validation: expected '{expected_token}'; received {received_tokens}")
assert received_tokens == {expected_token}, "Token mismatch!"
print("  ✓ All notifications carry the correct shared-secret token")

## Step 8 — Clean shutdown; Path 04 closure

Terminate all three subprocesses. Path 04 is now complete — seven shipped modules across the MCP and A2A territory.

## What's still ahead

- **OAuth2 cross-org token flows** — Module 6 sketched the security scheme; production deployments need a real auth issuer. Path 08 (Production Engineering, planned) will cover full integration.
- **Distributed tracing across the composition** — Lab 29 demonstrated OTel within one process. Cross-process trace correlation needs `traceparent` header forwarding across the A2A boundary; the SDK doesn't do this automatically yet.
- **Webhook receiver hardening** — HMAC token validation, replay protection via idempotency keys, key rotation. Robust webhook handling is its own substantial topic.
- **Multi-worker orchestration** — combine A2A delegation with Path 03's plan-and-execute pattern; one orchestrator dispatches to several specialist workers.

## Path 04 complete — 7 of 7 modules shipped

1. ✅ MCP foundations
2. ✅ Building an MCP server (+ Lab 25)
3. ✅ Building an MCP client (+ Lab 26)
4. ✅ MCP security threat model (+ Lab 27)
5. ✅ A2A foundations (+ Lab 28)
6. ✅ A2A endpoint at production depth (+ Lab 29)
7. ✅ MCP + A2A composition (+ Lab 30) — this lab

In [ ]:
# Clean shutdown of all three subprocesses

for name, proc in [("MCP server", mcp_proc),
                   ("A2A worker", worker_proc),
                   ("Webhook receiver", webhook_proc)]:
    proc.terminate()
    try:
        proc.wait(timeout=5.0)
        print(f"  ✓ {name} stopped (exit {proc.returncode})")
    except subprocess.TimeoutExpired:
        proc.kill()
        proc.wait()
        print(f"  ⚠ {name} force-killed")

print("\n✓ All subprocesses stopped cleanly.")

## Test yourself

Take the [MCP + A2A composition quiz](../../quizzes/foundations/mcp-a2a-composition.md) — 8 questions covering Module 7 and this lab. Topics: the asymmetric composition shape, atomic push-notification registration, `return_immediately=True` semantics, idempotency requirements for webhooks, the `ClientFactory`/`A2ACardResolver` surface, when composition is overkill, and the limits the composition does NOT solve.

This closes Path 04. The next path to engage with is whatever fits your current work — [Path 03 Multi-Agent Systems](../../learning-paths/03-multi-agent-systems/) for in-process orchestration; [Path 06 Evaluation & Observability](../../learning-paths/06-evaluation-observability/) for measurement; one of the architectural patterns directly via [`patterns/`](../../patterns/) for cross-cutting design ideas.